# Food Image Recognition & Calories Estimation (YOLO26m-cls)

This notebook is a real, notebook-first refactor for Food-101 classification using YOLO classification models plus an optional calorie metadata lookup.

Dataset source: https://www.kaggle.com/datasets/dansbecker/food-101

## Environment Setup

In [ ]:
import importlib
import subprocess
import sys

def ensure_package(import_name: str, pip_name: str | None = None) -> None:
    pkg = pip_name or import_name
    try:
        importlib.import_module(import_name)
    except Exception:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

ensure_package('kagglehub')
ensure_package('numpy')
ensure_package('pandas')
ensure_package('PIL', 'Pillow')
ensure_package('matplotlib')
ensure_package('sklearn', 'scikit-learn')
ensure_package('yaml', 'pyyaml')
ensure_package('ultralytics')
print('Dependencies are ready.')

## Imports and Configuration

In [ ]:
import json
import os
import random
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, top_k_accuracy_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

PROJECT_DIR = Path.cwd()
WORK_DIR = PROJECT_DIR / 'working' / 'food101'
RAW_DIR = WORK_DIR / 'raw'
PREP_DIR = WORK_DIR / 'prepared_cls'
ARTIFACTS_DIR = PROJECT_DIR / 'artifacts'

for d in [WORK_DIR, RAW_DIR, PREP_DIR, ARTIFACTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Optional row capping for local runtime while keeping real data workflow
MAX_TRAIN_PER_CLASS = 120
MAX_VAL_PER_CLASS = 60
MAX_EVAL_SAMPLES = 2000

print(f'Project dir: {PROJECT_DIR}')
print(f'Working dir: {WORK_DIR}')

## Real Dataset Download

In [ ]:
import kagglehub

if not os.getenv('KAGGLE_USERNAME') or not os.getenv('KAGGLE_KEY'):
    raise EnvironmentError('Missing Kaggle credentials. Set KAGGLE_USERNAME and KAGGLE_KEY in your environment.')

dataset_root = Path(kagglehub.dataset_download('dansbecker/food-101'))
print(f'Dataset root: {dataset_root}')

## Verify Files, Labels, and Splits

In [ ]:
images_dir = dataset_root / 'images'
meta_dir = dataset_root / 'meta'
train_txt = meta_dir / 'train.txt'
test_txt = meta_dir / 'test.txt'

if not images_dir.exists():
    raise FileNotFoundError(f'Missing images directory: {images_dir}')
if not meta_dir.exists():
    raise FileNotFoundError(f'Missing meta directory: {meta_dir}')
if not train_txt.exists() or not test_txt.exists():
    raise FileNotFoundError('Expected meta/train.txt and meta/test.txt files were not found.')

def parse_split(txt_path: Path, split_name: str) -> pd.DataFrame:
    rows = []
    with open(txt_path, 'r', encoding='utf-8') as f:
        for line in f:
            rel = line.strip()
            if not rel:
                continue
            label = rel.split('/')[0]
            img_path = images_dir / f'{rel}.jpg'
            rows.append({'split': split_name, 'label': label, 'rel': rel, 'image_path': str(img_path)})
    return pd.DataFrame(rows)

train_df = parse_split(train_txt, 'train')
val_df = parse_split(test_txt, 'val')
df = pd.concat([train_df, val_df], ignore_index=True)

if len(df) == 0:
    raise RuntimeError('Split files produced an empty dataframe.')

missing_files = (~df['image_path'].map(lambda p: Path(p).exists())).sum()
if missing_files > 0:
    raise RuntimeError(f'Found missing image files referenced by split metadata: {missing_files}')

duplicate_rels = df['rel'].duplicated().sum()

class_list = sorted(df['label'].unique().tolist())
if len(class_list) < 2:
    raise RuntimeError('Expected multiple classes but found fewer than 2.')

print(f'Total rows: {len(df)}')
print(f'Num classes: {len(class_list)}')
print(f'Duplicate relative IDs across combined splits: {duplicate_rels}')
print(df.groupby(['split', 'label']).size().groupby(level=0).describe())
df.head()

In [ ]:
# Sample integrity check on real image files
sample_paths = df['image_path'].sample(n=min(500, len(df)), random_state=SEED).tolist()
corrupt_count = 0
for p in sample_paths:
    try:
        with Image.open(p) as img:
            img.verify()
    except Exception:
        corrupt_count += 1

if corrupt_count > 0:
    raise RuntimeError(f'Corrupt images found in sampled integrity check: {corrupt_count}')

print('Sampled image integrity check passed.')

## Prepare YOLO Classification Directory

In [ ]:
for split_name in ['train', 'val']:
    for cls in class_list:
        (PREP_DIR / split_name / cls).mkdir(parents=True, exist_ok=True)

def cap_by_class(source_df: pd.DataFrame, limit_per_class: int) -> pd.DataFrame:
    capped_parts = []
    for cls in class_list:
        part = source_df[source_df['label'] == cls].copy()
        if len(part) > limit_per_class:
            part = part.sample(n=limit_per_class, random_state=SEED)
        capped_parts.append(part)
    return pd.concat(capped_parts, ignore_index=True)

train_small = cap_by_class(train_df, MAX_TRAIN_PER_CLASS)
val_small = cap_by_class(val_df, MAX_VAL_PER_CLASS)

for _, row in train_small.iterrows():
    src = Path(row['image_path'])
    dst = PREP_DIR / 'train' / row['label'] / src.name
    if not dst.exists():
        shutil.copy2(src, dst)

for _, row in val_small.iterrows():
    src = Path(row['image_path'])
    dst = PREP_DIR / 'val' / row['label'] / src.name
    if not dst.exists():
        shutil.copy2(src, dst)

print(f'Prepared train rows: {len(train_small)}')
print(f'Prepared val rows: {len(val_small)}')
print(f'Prepared data root: {PREP_DIR}')

In [ ]:
def count_images_in_split(split_dir: Path) -> int:
    return sum(1 for p in split_dir.rglob('*') if p.suffix.lower() in {'.jpg', '.jpeg', '.png'})

n_train_files = count_images_in_split(PREP_DIR / 'train')
n_val_files = count_images_in_split(PREP_DIR / 'val')

if n_train_files == 0 or n_val_files == 0:
    raise RuntimeError('Prepared YOLO cls folders are empty.')

print(f'Train image files: {n_train_files}')
print(f'Val image files: {n_val_files}')

## Real Data Visualization

In [ ]:
viz_df = train_small.sample(n=min(9, len(train_small)), random_state=SEED).reset_index(drop=True)
fig, axes = plt.subplots(3, 3, figsize=(12, 10))
for i in range(len(viz_df)):
    row = viz_df.iloc[i]
    img = Image.open(row['image_path']).convert('RGB')
    axes.flatten()[i].imshow(img)
    axes.flatten()[i].set_title(row['label'])
    axes.flatten()[i].axis('off')
for j in range(len(viz_df), 9):
    axes.flatten()[j].axis('off')
plt.tight_layout()
sample_grid_path = ARTIFACTS_DIR / 'sample_grid.png'
plt.savefig(sample_grid_path, dpi=140)
plt.show()

## Train YOLO26m-cls (with practical fallback)

In [ ]:
from ultralytics import YOLO

weights_candidates = ['yolo26m-cls.pt', 'yolo11m-cls.pt', 'yolov8m-cls.pt']
selected_weights = None
model = None

for w in weights_candidates:
    try:
        model = YOLO(w)
        selected_weights = w
        break
    except Exception:
        continue

if selected_weights is None:
    raise RuntimeError('Could not load a YOLO classification checkpoint from known candidates.')

print(f'Selected checkpoint: {selected_weights}')

train_result = model.train(
    data=str(PREP_DIR),
    epochs=2,
    imgsz=160,
    batch=64,
    project=str(ARTIFACTS_DIR / 'yolo_runs'),
    name='food_cls',
    seed=SEED,
    verbose=False
)

best_model_path = Path(model.trainer.best)
print(f'Best model: {best_model_path}')

## Real Evaluation

In [ ]:
best_model = YOLO(str(best_model_path))

label_to_idx = {name: i for i, name in enumerate(sorted(class_list))}
idx_to_label = {v: k for k, v in label_to_idx.items()}

eval_df = val_small.copy().reset_index(drop=True)
if len(eval_df) > MAX_EVAL_SAMPLES:
    eval_df = eval_df.sample(n=MAX_EVAL_SAMPLES, random_state=SEED).reset_index(drop=True)

y_true = []
y_pred = []
y_prob_rows = []

for _, row in eval_df.iterrows():
    pred = best_model.predict(source=row['image_path'], verbose=False)[0]
    probs = pred.probs.data.cpu().numpy()
    y_true.append(label_to_idx[row['label']])
    y_pred.append(int(np.argmax(probs)))
    y_prob_rows.append(probs)

y_prob = np.vstack(y_prob_rows)

acc = accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average='macro')
top5 = top_k_accuracy_score(y_true, y_prob, k=5, labels=list(range(len(class_list))))
cm = confusion_matrix(y_true, y_pred)

print(f'Eval samples: {len(eval_df)}')
print(f'Accuracy: {acc:.4f}')
print(f'Macro F1: {macro_f1:.4f}')
print(f'Top-5 accuracy: {top5:.4f}')
print(classification_report(y_true, y_pred, labels=list(range(len(class_list))), target_names=sorted(class_list), zero_division=0))

In [ ]:
# Plot a compact confusion matrix for the 20 most frequent classes in eval_df
top_classes = eval_df['label'].value_counts().head(20).index.tolist()
top_idxs = [label_to_idx[c] for c in top_classes]

mask = np.isin(y_true, top_idxs)
y_true_top = np.array(y_true)[mask]
y_pred_top = np.array(y_pred)[mask]

cm_top = confusion_matrix(y_true_top, y_pred_top, labels=top_idxs)

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(cm_top, cmap='Blues')
ax.set_title('Confusion Matrix (Top 20 Frequent Eval Classes)')
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_xticks(range(len(top_classes)))
ax.set_yticks(range(len(top_classes)))
ax.set_xticklabels(top_classes, rotation=90)
ax.set_yticklabels(top_classes)
plt.tight_layout()
cm_path = ARTIFACTS_DIR / 'confusion_matrix_top20.png'
plt.savefig(cm_path, dpi=140)
plt.show()

## Optional Calories Metadata Lookup

This is an optional post-classification lookup using approximate per-serving calories for selected Food-101 classes.
Classes not in the lookup are marked as unknown to avoid fake precision.

In [ ]:
calorie_lookup = {
    'apple_pie': 296,
    'beef_tartare': 246,
    'caesar_salad': 190,
    'cheesecake': 321,
    'chicken_curry': 243,
    'chicken_wings': 290,
    'club_sandwich': 220,
    'donuts': 452,
    'french_fries': 312,
    'fried_rice': 238,
    'greek_salad': 170,
    'hamburger': 295,
    'hot_dog': 290,
    'ice_cream': 207,
    'lasagna': 257,
    'omelette': 154,
    'pancakes': 227,
    'pizza': 266,
    'ramen': 436,
    'spaghetti_bolognese': 160,
    'steak': 271,
    'sushi': 150,
    'waffles': 291
}

pred_labels = [idx_to_label[i] for i in y_pred]

pred_df = eval_df.copy()
pred_df['true_label'] = [idx_to_label[i] for i in y_true]
pred_df['pred_label'] = pred_labels
pred_df['estimated_calories_per_serving'] = pred_df['pred_label'].map(calorie_lookup)
pred_df['calorie_lookup_available'] = pred_df['estimated_calories_per_serving'].notna().astype(int)

pred_csv_path = ARTIFACTS_DIR / 'eval_predictions_with_calories.csv'
pred_df.to_csv(pred_csv_path, index=False)

coverage = pred_df['calorie_lookup_available'].mean()
print(f'Calorie lookup coverage on eval predictions: {coverage:.2%}')
pred_df[['image_path', 'true_label', 'pred_label', 'estimated_calories_per_serving']].head(10)

## Save Real Outputs

In [ ]:
metrics = {
    'dataset': 'dansbecker/food-101',
    'dataset_url': 'https://www.kaggle.com/datasets/dansbecker/food-101',
    'num_classes_total': int(len(class_list)),
    'num_rows_total': int(len(df)),
    'num_train_rows_original': int(len(train_df)),
    'num_val_rows_original': int(len(val_df)),
    'num_train_rows_prepared': int(len(train_small)),
    'num_val_rows_prepared': int(len(val_small)),
    'selected_weights': selected_weights,
    'best_model_path': str(best_model_path),
    'eval_samples': int(len(eval_df)),
    'accuracy': float(acc),
    'macro_f1': float(macro_f1),
    'top5_accuracy': float(top5),
    'max_train_per_class': int(MAX_TRAIN_PER_CLASS),
    'max_val_per_class': int(MAX_VAL_PER_CLASS),
    'max_eval_samples': int(MAX_EVAL_SAMPLES),
    'calorie_lookup_coverage': float(coverage)
}

metrics_path = ARTIFACTS_DIR / 'metrics.json'
with open(metrics_path, 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2)

manifest = {
    'dataset_root': str(dataset_root),
    'prepared_data_root': str(PREP_DIR),
    'metrics_json': str(metrics_path),
    'predictions_csv': str(pred_csv_path),
    'sample_grid_png': str(sample_grid_path),
    'confusion_matrix_png': str(cm_path)
}
manifest_path = ARTIFACTS_DIR / 'artifact_manifest.json'
with open(manifest_path, 'w', encoding='utf-8') as f:
    json.dump(manifest, f, indent=2)

print('Saved outputs:')
print(metrics_path)
print(pred_csv_path)
print(sample_grid_path)
print(cm_path)
print(manifest_path)

## Honest Limitations

- The calorie lookup is approximate and only available for a subset of classes.
- Training here uses capped per-class samples and short epochs for practical local runtime.
- Stronger final quality requires longer training and full Food-101 scale runs.